# Fork the ruler, not the model

*By [Georgy Mamarin](https://www.kaggle.com/georgymamarin)*

This notebook used to be called *Where the Wall Is*, then *Stop reforking*. There is no single wall. There is a part of the error you can recover from the data you are handed, and a tail you cannot model your way out of — and in the first version I had the irreducible tail mislabeled as the floor of the whole task.

Two things get forked on a leaderboard: the model, and the ruler you measure it with. This notebook is the ruler — a small harness ([§9](#9)) you point at your own predictions to read your floor, your tail, and whether a feature survives leaving a whole group out. The wellbore task is the worked example; the harness runs on any grouped regression.

Start with the numbers that anchor everything. The carry-last-TVT baseline scores **15.883** on the public board. The forks cluster around **7.2**; the most-forked reference base at the time of writing is "ROGII LB7295 Public Rebuild" (~400 votes at score 7.15). The board heads sit at **~5.3–5.5** (the top three as of July 3 — the board will keep moving in the final month, and neither the ladder nor the argument depends on the exact head score). The question worth asking before you fork anything: of that gap from 15.9 down to ~5.3, how much is recoverable, and how much is geology? One well first, to make the hard part concrete.

In [ ]:
# ---- one real worst-decile train well: the gamma-ray's best fit lands at the WRONG depth ----
import os, glob, numpy as np, pandas as pd
import matplotlib.pyplot as plt

def _draw_hero():
    def _train():
        for c in ["/kaggle/input/competitions/rogii-wellbore-geology-prediction/train",
                  "/kaggle/input/rogii-wellbore-geology-prediction/train", "data/train"]:
            if os.path.isdir(c): return c
        g = glob.glob("/kaggle/input/**/train", recursive=True); return g[0] if g else "data/train"
    T = _train()
    G = np.arange(-20.0, 20.01, 0.5); TR = np.abs(G) <= 2.5
    Cn = dict(red="#D55E00", ink="#1a1a1a", grey="#999999")
    def rfit(s, y, d=6):
        if len(s) < d+2: return y.copy()
        c = np.polyfit(s, y, d)
        for _ in range(3):
            r = y - np.polyval(c, s); sc = np.median(np.abs(r))*1.4826 + 1e-6
            c = np.polyfit(s, y, d, w=1.0/(1.0+(r/(2*sc))**2))
        return np.polyval(c, s)
    def sm(a, k=5): return pd.Series(a).rolling(k, center=True, min_periods=1).mean().to_numpy()
    def mis(gr, true, tv, tg):
        v = np.isfinite(gr)
        if v.sum() < 20: return None
        tgt = np.interp(true, tv, tg, left=tg[0], right=tg[-1])
        coef, *_ = np.linalg.lstsq(np.vstack([gr[v], np.ones(v.sum())]).T, tgt[v], rcond=None)
        cal = gr*coef[0] + coef[1]
        return np.array([np.mean((cal[v]-np.interp(true+dz, tv, tg, left=tg[0], right=tg[-1])[v])**2) for dz in G])
    fs = sorted(glob.glob(f"{T}/*__horizontal_well.csv"))
    rng = np.random.RandomState(42); fs = [fs[i] for i in rng.permutation(len(fs))[:160]]
    Ws = []
    for f in fs:
        try: hw = pd.read_csv(f, usecols=["MD","GR","TVT","TVT_input"])
        except Exception: continue
        tvi = hw["TVT_input"].to_numpy(float); tv = hw["TVT"].to_numpy(float)
        GR = hw["GR"].to_numpy(float); MD = hw["MD"].to_numpy(float)
        ei = np.flatnonzero(np.isnan(tvi) & np.isfinite(tv)); kn = np.flatnonzero(~np.isnan(tvi))
        if len(ei) < 50 or len(kn) < 50: continue
        twf = f.replace("__horizontal_well.csv","__typewell.csv")
        if not os.path.exists(twf): continue
        tw = pd.read_csv(twf); ttv = tw["TVT"].to_numpy(float); ttg = tw["GR"].to_numpy(float)
        o = np.argsort(ttv); ttv, ttg = ttv[o], ttg[o]; _, ix = np.unique(ttv, return_index=True); ttv, ttg = ttv[ix], ttg[ix]
        s = (MD[ei]-MD[kn[-1]])/max(MD[-1]-MD[kn[-1]], 1e-6)
        true = tv[ei]; gr = pd.Series(GR[ei]).interpolate(limit_direction="both").to_numpy()
        Ws.append(dict(true=true, gr=gr, tv=ttv, tg=ttg, sse=float(np.sum((true-rfit(s, true, 6))**2))))
    if not Ws: return
    order = np.argsort([w["sse"] for w in Ws])[::-1]
    worst = [Ws[i] for i in order[:max(1, len(Ws)//8)]]
    win = (G>=-18)&(G<=18); cands = []
    for w in worst:
        c = mis(w["gr"], w["true"], w["tv"], w["tg"])
        if c is None: continue
        cn = sm(c/c.min(), 5)
        vt = cn[TR].min(); dzt = G[TR][int(np.argmin(cn[TR]))]
        gi = int(np.argmin(cn)); gdz = G[gi]; gval = cn[gi]
        cands.append(dict(cn=cn, vt=vt, dzt=dzt, gdz=gdz, gval=gval,
                          rough=np.mean(np.abs(np.diff(cn[win], 2))), gap=vt/max(gval,1e-9), peak=cn[win].max()))
    if not cands: return
    good = [c for c in cands if abs(c["gdz"])>=6 and 1.07<=c["gap"]<=1.9 and abs(c["gdz"])<=17 and c["vt"]<1.8 and c["peak"]<4.0]
    if not good: good = [c for c in cands if abs(c["gdz"])>=4 and c["gap"]>=1.04]
    if not good: good = sorted(cands, key=lambda c:-abs(c["gdz"]))[:1]
    good.sort(key=lambda c: c["rough"])
    h = good[0]; dzt, vt, dzd, vd = h["dzt"], h["vt"], h["gdz"], h["gval"]

    fig, ax = plt.subplots(figsize=(9.6, 4.5)); fig.patch.set_facecolor("white"); ax.set_facecolor("white")
    ax.plot(G, h["cn"], color=Cn["red"], lw=2.8, zorder=5, solid_capstyle="round")
    ytop = max(vt+1.05, 2.7); ax.set_ylim(0.95, ytop)
    ax.axhline(vd, color=Cn["grey"], lw=1, ls=(0,(4,4)), alpha=.6, zorder=1)
    lo, hi = sorted([dzt, dzd]); ax.set_xlim(lo-6, hi+6.5)   # frame the two markers, trim dead space
    ax.scatter([dzt],[vt], s=96, color=Cn["ink"], zorder=6)
    ax.scatter([dzd],[vd], s=125, facecolor="none", edgecolor=Cn["red"], linewidth=2.6, zorder=6)
    ysky = ytop - 0.34
    def lab(dz, val, l1, l2, col):
        ax.annotate("", xy=(dz, val+0.05), xytext=(dz, ysky-0.02), arrowprops=dict(arrowstyle="-", color="#999", lw=1))
        ax.text(dz, ysky+0.14, l1, ha="center", va="bottom", fontsize=12, color=col, fontweight="700")
        ax.text(dz, ysky-0.0, l2, ha="center", va="bottom", fontsize=10.5, color=Cn["ink"])
    lab(dzt, vt, "the true depth", "(not the lowest fit)", Cn["ink"])
    lab(dzd, vd, "the best GR match", "— a wrong depth", Cn["red"])
    yb = vd - 0.085
    ax.plot([dzt, dzt], [yb, vt], color="#888", lw=1, ls=(0, (3, 3)), alpha=.55, zorder=2)  # dropline: true-depth point -> the distance brace, so "N ft off" reads as point-to-point
    ax.plot([lo,hi],[yb,yb], color="#888", lw=1.2, zorder=4)
    for xx in (lo,hi): ax.plot([xx,xx],[yb-0.022,yb+0.022], color="#888", lw=1.2, zorder=4)
    ax.text((lo+hi)/2, yb, f"≈ {abs(dzd-dzt):.0f} ft off the true depth", ha="center", va="center",
            fontsize=9.5, color="#555", bbox=dict(facecolor="white", edgecolor="none", pad=1.4), zorder=5)
    ax.set_xlabel("vertical shift from the true depth   (ft)", fontsize=10.5)
    ax.set_ylabel("GR misfit   (own best = 1.0)", fontsize=10.5)
    for sp in ("top","right"): ax.spines[sp].set_visible(False)
    ax.tick_params(labelsize=9.5); ax.grid(alpha=.18)
    fig.suptitle("On these wells, the gamma-ray's best fit is the wrong depth.", x=0.012, y=0.965,
                 ha="left", fontsize=16, fontweight="800", color=Cn["ink"])
    fig.text(0.012, 0.875, "These worst ~10% of wells carry ~40% of the error — the rest is recoverable, and this notebook maps which is which.",
             ha="left", fontsize=10.5, color="#555")
    fig.text(0.012, 0.022, "a real worst-decile train well, misfit lightly smoothed for display  ·  souldrive's NCC cost surface agrees on 773 wells (tie-break r = +0.054)",
             ha="left", fontsize=8.3, color=Cn["grey"])
    fig.subplots_adjust(left=0.078, right=0.985, top=0.78, bottom=0.215)
    plt.show()

_draw_hero()

That is the irreducible part of the map. The short version of the whole map:

- **Recoverable** — a per-well offset plus a piecewise dip, read by matching the horizontal well's gamma-ray to the typewell. The easier wells get to roughly 3–5&nbsp;ft per-well this way. Public forks (~7.2) have captured the offset and the dominant slope; the heads capture some of the wiggle on top.
- **Irreducible** — the bimodal datum in the figure above, on a minority of wells. The Eagle Ford is rhythmically bedded, so the GR lines up with the typewell at two stratigraphic positions about one bundle apart (roughly ±15&nbsp;ft), and the truth is close to a coin-flip. In my train-side measurements, the worst ~10% of wells carry about 40% of the squared error.

Before the science, two corrections, because I got two things wrong in the first version and would rather state them plainly than bury them.

**Correction 1 — there is no leak.** I claimed the public leaderboard was scored on three train-copy wells, so it read generously. It is not: that local `data/test/` folder is example data, replaced by the real hidden test (~200 wells) at scoring — confirmed on the Data page and by Ioannis M (rank 28) and the host. The public LB is ~26% of those hidden wells, scored honestly; the private set decides medals. I retract the leak entirely (the byte-identical example wells and the ~0.007 ANCC-cheat coincidence I misread are detailed in §10).

**Correction 2 — ~10&nbsp;ft is not a wall.** I framed ~10&nbsp;ft as the task's floor. It is not. The ~10 was *my selector's* ceiling — a per-well GR particle filter plus beam search — under a deliberately hard split that leaves whole fields out (block-CV ~10.9). Two things on the board already prove ~10 is not the floor: the heads sit far below it, and my own smooth oracle (§2) sits at ~3.0. Tucker (rank 2) described reaching about 5 per-well using per-well data only — a cross-validation figure on train, not a leaderboard score, so §7's CV→LB caveat applies to it too — but even so, at least one head is simply matching GR better than my selector did. The gap is method quality, not a leak.

This is a diagnostic, not a submission: it scores nothing, runs end-to-end on the public competition data alone, and is built to be forked. One expectation that shaped how I wrote it — the most-upvoted notebook here (a DWT one, ~630 votes) scores about 9.25, so on this task votes follow clarity more than score. Treat this as a map of the problem, not a leaderboard lever.

**Skimming? Three things.** The recoverable floor is ~3–5&nbsp;ft per-well, well below the ~7.2 fork cluster ([§2](#2)). The hard part is a bimodal datum on the ~10% of wells that carry ~40% of the error — estimate `p` and predict the posterior mean rather than commit to a mode ([§5](#5)). And before you trust a gain, fork the harness and read your own ceiling and seed band off it ([§9](#9)).

## Contents
1. [The task in one paragraph](#1)
2. [The floor: a calibrated oracle ladder](#2)
3. [The error collapses to offset plus a piecewise dip](#3)
4. [Reading the dip: GR-to-typewell matching has a quality ceiling](#4)
5. [The irreducible tail: a bimodal datum](#5)
6. [The leave-field-out result, in proportion](#6)
7. [Traps that look like progress](#7)
8. [What to do if you are stuck near the cluster](#8)
9. [Fork the ruler, not the model](#9)
10. [Design choices and limitations](#10)
11. [References and a question for you](#11)

<a id="setup"></a>
### Setup

Everything below runs on the **public competition data only** — no private artifacts — so a fork runs end-to-end. Seeded; a 250-well subsample keeps the run to a couple of minutes (the full 773 wells give the same picture).

In [ ]:
import os, glob, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
warnings.filterwarnings("ignore")

SEED = 42; np.random.seed(SEED)
N_WELLS = 250                         # subsample for runtime; full 773 ~ same result
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True,
                     "grid.alpha": .25, "axes.axisbelow": True})
# colorblind-safe (Okabe–Ito)
C = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00",
         grey="#999999", purple="#CC79A7")

def find_train():
    for c in ["/kaggle/input/competitions/rogii-wellbore-geology-prediction/train",
              "/kaggle/input/rogii-wellbore-geology-prediction/train", "data/train"]:
        if os.path.isdir(c): return c
    g = glob.glob("/kaggle/input/**/train", recursive=True)   # robust mount resolve
    return g[0] if g else "data/train"

TRAIN = find_train()
files = sorted(glob.glob(f"{TRAIN}/*__horizontal_well.csv"))
rng = np.random.RandomState(SEED)
files = [files[i] for i in rng.permutation(len(files))[:N_WELLS]]
rmse = lambda a: float(np.sqrt(np.mean(np.asarray(a, float) ** 2)))
print(f"train dir: {TRAIN}")
print(f"wells in sample: {len(files)} / {len(glob.glob(f'{TRAIN}/*__horizontal_well.csv'))}")

<a id="1"></a>
## 1. The task in one paragraph

Each well is a horizontal trajectory with logs `MD, X, Y, Z, GR`, a known `TVT_input` up to a prediction-start point, and a vertical **typewell** (GR vs TVT). We predict `TVT` on the hidden eval tail; on the eval rows `X/Y/Z` are known and `TVT` is hidden. The metric is pooled row-wise RMSE over the eval rows.

A few community-established facts fix the problem; I take them as given.

The first is that the target reduces to a surface. `TVT = ANCC(surface) − Z + b_well`, with `b_well` constant per well (std 0.0064&nbsp;ft). `Z` is known on the tail and the offset is pinned by the heel, so predicting `TVT` is reading the formation surface along the lateral. One caution about reading that identity through a correlation: pooled across all wells `ANCC` and `TVT` correlate at about −0.94, which a couple of EDAs have read as a near-leaky column, but within a single well the same correlation flips sign to roughly +0.26. The sign flip is Simpson's paradox — the pooled number is dominated by the between-well offsets, not the within-well relationship — so the strong pooled correlation is not a shortcut to exploit. The identity above, with `b_well` pinned per well, is the honest way to use it.

The second is that the six formation columns are not six surfaces. franticXu showed that `ANCC, ASTNU, ASTNL, EGFDU, EGFDL, BUDA` collapse to one base surface plus five constant offsets — one degree of freedom, with layer thickness std under 0.01&nbsp;ft along each well. They are derived from the typewell's TVT boundaries (in 97.5% of wells the typewell thickness equals the horizontal-well thickness), not measured 3D surfaces. One point that trips people up: `TVT` here is a *cumulative* vertical distance, like TVD, rather than the thickness of a single layer. The test wells ship without these formation columns, so they have to be reconstructed, and a few wells have a truncated or absent ANCC.

The third is what "matching" even means. Tabish's breakdown puts it cleanly: the typewell is a GR-vs-TVT lookup, and you read your stratigraphic position by matching the horizontal well's GR trace against it. The layers run roughly parallel, and many wells share subsequences of a master typewell sequence. So the target is not one number; it is a piecewise dip, what Tabish calls "parallel jagged lines."

The rest of the notebook maps where that surface is recoverable from the data and where it is not.

In [ ]:
# load eval-row truth + geometry once; reused throughout
def load(f):
    hw = pd.read_csv(f)
    tvi, tv = hw["TVT_input"].to_numpy(float), hw["TVT"].to_numpy(float)
    Z, MD, GR = hw["Z"].to_numpy(float), hw["MD"].to_numpy(float), hw["GR"].to_numpy(float)
    X, Y = hw["X"].to_numpy(float), hw["Y"].to_numpy(float)
    ev = np.isnan(tvi) & np.isfinite(tv); ei = np.flatnonzero(ev)
    kn = np.flatnonzero(~np.isnan(tvi))
    if len(ei) < 50 or len(kn) < 50: return None
    li = kn[-1]
    s = (MD[ei] - MD[li]) / max(MD[-1] - MD[li], 1e-6)        # normalized tail position 0..1
    return dict(wid=os.path.basename(f)[:8], file=f, true=tv[ei], s=s, Z=Z[ei], MD=MD[ei],
                last=tvi[li], surf=tv[ei] + Z[ei], X=X, Y=Y, GR=GR, ei=ei, kn=kn, li=li,
                tvi=tvi, hwZ=Z, hwMD=MD)
W = [w for w in (load(f) for f in files) if w is not None]
print(f"usable wells: {len(W)}")

<a id="2"></a>
## 2. The floor: a calibrated oracle ladder

Before chasing a better model, ask the cheaper question: if you already knew the answer, what is the best each modeling assumption could do? Each rung below uses the *true* tail to fit the simplest possible shape, then reports its pooled RMSE. That is a ceiling no model *restricted to that shape* can beat — and the ladder turns out to be anchored to the real board, which is the point.

- **flat** — carry the last known TVT (the only fully-legal rung here; a sanity floor).
- **constant** — the best single per-well offset.
- **line** — the best per-well straight line vs measured depth.
- **smooth** — a robust degree-6 fit (captures the wiggle too).

In [ ]:
def robfit(s, y, deg):
    if len(s) < deg + 2: return y.copy()
    c = np.polyfit(s, y, deg)
    for _ in range(3):
        r = y - np.polyval(c, s); sc = np.median(np.abs(r)) * 1.4826 + 1e-6
        c = np.polyfit(s, y, deg, w=1.0 / (1.0 + (r / (2 * sc)) ** 2))
    return np.polyval(c, s)

E = {k: [] for k in ["flat", "constant", "line", "smooth"]}
for w in W:
    t, s = w["true"], w["s"]
    E["flat"].append(t - w["last"])
    E["constant"].append(t - t.mean())
    E["line"].append(t - np.polyval(np.polyfit(s, t, 1), s))
    E["smooth"].append(t - robfit(s, t, 6))
ladder = {k: rmse(np.concatenate(v)) for k, v in E.items()}
_tbl = pd.DataFrame({"Oracle RMSE (ft)": [ladder[k] for k in ["flat","constant","line","smooth"]],
                     "What it captures": ["carry the last known value", "best single offset", "best straight line", "adds the wiggle"]},
                    index=["flat (legal)", "constant offset", "per-well line", "smooth (deg-6)"])
_tbl.index.name = "Assumption (oracle)"

# per-well figures used later (measured here, not borrowed):
line_w   = np.array([rmse(e) for e in E["line"]])      # per-well RMSE under the line oracle
smooth_w = np.array([rmse(e) for e in E["smooth"]])    # ...and under the smooth oracle
sse_w    = np.array([float(np.sum(e**2)) for e in E["smooth"]])
_ord = np.argsort(sse_w)[::-1]; _k10 = max(1, len(sse_w)//10)
worst10_share = sse_w[_ord[:_k10]].sum() / sse_w.sum()
print(f"wells already under 5 ft per-well:  line oracle {np.mean(line_w<5):.0%}   smooth oracle {np.mean(smooth_w<5):.0%}")
print(f"worst 10% of wells carry {worst10_share:.0%} of the smooth-oracle squared error  (the tail; see §5)")
_tbl.style.format({"Oracle RMSE (ft)": "{:.2f}"}).background_gradient(subset=["Oracle RMSE (ft)"], cmap="Greens_r")

**Table 1 — the oracle ladder.** Pooled RMSE over 250 train wells; each rung fits the simplest shape to the *true* tail, so it is an unreachable ceiling *for that assumption*, not a model score.

In [ ]:
fig, ax = plt.subplots(figsize=(7.8, 3.6))
names = ["flat\n(legal)", "constant\noffset", "per-well\nline", "smooth\n(deg-6)"]
vals = [ladder[k] for k in ["flat", "constant", "line", "smooth"]]
bars = ax.bar(names, vals, color=[C["grey"], C["orange"], C["blue"], C["green"]], width=.62)
for b, v in zip(bars, vals): ax.text(b.get_x()+b.get_width()/2, v+.15, f"{v:.1f}", ha="center", fontsize=10)
# reference lines — CITED from the REAL public LB (honestly scored on the hidden wells), NOT recomputed here.
ax.axhspan(5.4, 7.4, color="#cccccc", alpha=.18, zorder=0)
ax.axhline(15.883, ls="--", lw=1.3, color=C["grey"],   label="carry-last baseline 15.9  (public LB)")
ax.axhline(7.2,    ls="--", lw=1.3, color=C["red"],    label="public best fork ~7.2  (public LB)")
ax.axhline(5.3,    ls="--", lw=1.3, color=C["purple"], label="LB heads ~5.3  (public LB, Jul 3)")
ax.legend(loc="upper right", fontsize=8.5, framealpha=.95)
ax.text(0.02, 0.55, "bars: train wells, pooled RMSE\ndashed: real public LB, cited",
        transform=ax.transAxes, va="top", ha="left", fontsize=8, color="#333",
        bbox=dict(boxstyle="round,pad=0.3", fc="#f7f7f7", ec="#bbbbbb", alpha=.95))
ax.set_ylabel("RMSE (ft, lower = better)"); ax.set_ylim(0, 17.5)
ax.set_title("Oracle ladder (train, pooled RMSE)  vs  the real public LB")
plt.tight_layout(); plt.show()

**Read it like this.** Carrying the last value is hopeless (~16). The live carry-last baseline scores **15.883**. Same procedure, different well sets — my flat oracle on a 250-well train sample, the baseline on the hidden public wells — so I read the agreement as a sanity check that my ladder is on the board's scale, not as proof of anything finer.

From there it descends. One offset per well already cuts it to ~9. The per-well **line** oracle (~6.6) sits just below the fork cluster (~7.2). I read that proximity as the forks capturing close to the offset-plus-line content and not much past it, but that is an inference from where the scores land, not a decomposition of what the forks actually do. The board **heads** (~5.3–5.5 as of early July) reach *below* the line oracle, which is the cleanest evidence in the chart that they are modeling more than a line — some of the wiggle and the dip changes. And the smooth oracle at ~3.0 says there is still recoverable structure sitting under even the heads.

One fence, so the two kinds of marker are not confused. The **bars** are train-fit ceilings — pooled RMSE on 250 train wells, each unreachable for its assumption because each needed the truth to fit. The **dashed lines** are live, honestly-scored public-LB markers. Different categories, same RMSE scale. The ordering is what carries the argument: forks near the line oracle, heads reaching into the wiggle, recoverable structure still below. The next sections are about what "reaching into the wiggle" requires.

One more anchor, added after wharekawa reproduced this ladder in the discussion to the last digit: on the full 773 wells it reads 9.04 / 6.70 / 3.05 with flat at 15.91 — same shape, same story. A reproduction is the strongest check a harness can get.

<a id="3"></a>
## 3. The error collapses to offset plus a piecewise dip

Take the surface along each lateral, `ANCC = TVT + Z`, and fit a straight line in measured depth. How much is left?

In [ ]:
lin_r2, wiggle, slopes = [], [], []
for w in W:
    surf, s = w["surf"], w["s"]
    lin = np.polyval(np.polyfit(s, surf, 1), s)
    ss_res = np.sum((surf - lin) ** 2); ss_tot = np.sum((surf - surf.mean()) ** 2)
    lin_r2.append(1 - ss_res / max(ss_tot, 1e-9))
    wiggle.append(np.std(surf - lin))
    slopes.append(np.polyfit(w["s"], w["true"] - w["last"], 1)[0])   # drift slope vs flat baseline
print(f"surface linear-fit R^2 : median {np.median(lin_r2):.3f}   ({np.mean(np.array(lin_r2)>0.9):.0%} of wells > 0.90)")
print(f"residual wiggle (std)  : median {np.median(wiggle):.2f} ft")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.3))
ax[0].hist(lin_r2, bins=30, color=C["blue"], alpha=.85)
ax[0].axvline(np.median(lin_r2), color=C["red"], lw=1.5)
ax[0].set_title("Surface is a line"); ax[0].set_xlabel("per-well linear-fit R²  (ANCC vs MD)")
ax[0].set_ylabel("wells"); ax[0].text(.05,.9,f"median {np.median(lin_r2):.3f}",transform=ax[0].transAxes,color=C["red"])
ax[1].hist(np.clip(wiggle,0,15), bins=30, color=C["green"], alpha=.85)
ax[1].axvline(np.median(wiggle), color=C["red"], lw=1.5)
ax[1].set_title("…and the wiggle around it is small"); ax[1].set_xlabel("residual after the line (ft, std)")
ax[1].set_ylabel("wells"); ax[1].text(.4,.9,f"median {np.median(wiggle):.1f} ft",transform=ax[1].transAxes,color=C["red"])
plt.tight_layout(); plt.show()

For the typical well the surface is a straight line to within a few feet over the whole lateral: median R² ≈ 0.99. The exceptions are the faulted minority, with steps of 15–30&nbsp;ft, and they are exactly the piecewise tail §5 is about, so don't let the median hide them. The line claim is about the middle of the distribution, not its ends.

That collapses the problem, but not down to a single number. Once the per-well offset is set by the heel, what is left is the dip, and the dip varies along the well. Following Tabish, `TVT + Z` and the layers are piecewise continuous linear functions, the "parallel jagged lines": a dominant slope plus the break points where it changes. The offset is pinned by the heel; the dip and its breaks are what you have to read. The breaks come from the same sub-seismic faults — they belong to the hard tail (§5), not to noise you should smooth away.

This is the recoverable part of the map. The offset plus the dominant slope is what the public forks (~7.2) have captured; the heads pull in some of the dip changes and the wiggle on top. The cell in §2 prints how many wells already sit under 5&nbsp;ft per-well from a clean smooth fit — a sizable chunk, which is why the room to improve is on the median wells. So the gap below the cluster is a matching problem: read the dip off the typewell well enough, on more wells. The next section is about that match.

<a id="4"></a>
## 4. Reading the dip: GR-to-typewell matching has a quality ceiling

Geosteering works because the gamma-ray log fingerprints the rock: match the well's `GR` to the typewell's `GR`-vs-`TVT` profile and you know your vertical position. The question is how far that match gets you, and what the bottleneck is.

First, the legality, because it decides what "matching" is even allowed to use. Settled in the Tiago thread: using the full per-well GR trace, including rows ahead of the eval row (the within-well "look-ahead"), is legal. This is post-drilling analysis, and the rule there was that everything in the data we are given is fair game unless explicitly forbidden. So matching your whole GR trace against the typewell, which is what the top solvers do, is fair.

And it reaches further than I gave it credit for. Tucker (rank 2) described reaching about 5 per-well using per-well data only — no cross-well features, no spatial structure, no external tops. I am reading that off his post rather than a shared notebook, and it is a train-CV figure rather than a leaderboard score, so I will not lean on it harder than that. But even as one data point it scopes the task: for at least the front of the field, the live lever is per-well GR-matching quality, and the cross-field slope transfer I spent a section proving hard (§6) is not required to get there.

So why doesn't the match trivially nail it? There is a real degeneracy, and I measure it with a **shift scan**. For each well, slide a candidate vertical offset `dz` and ask where the GR-misfit against the typewell is smallest, under three calibrations of the GR gain and offset:

- **oracle** — calibrate at the *true* TVT (illegal; an upper bound on what GR knows).
- **legal** — calibrate at a legitimate flat-surface guess (`last_known + (Z_heel − Z)`, known quantities only).
- **shuffle** — calibrate at a shuffled TVT (a random control).

In [ ]:
GRID = np.arange(-20.0, 20.01, 0.5)            # candidate vertical shifts dz; 0 = the truth
def twgr(tvt, tv, tg): return np.interp(tvt, tv, tg, left=tg[0], right=tg[-1])
def load_tw(w):
    twf = w["file"].replace("__horizontal_well.csv", "__typewell.csv")
    if not os.path.exists(twf): return None, None
    tw = pd.read_csv(twf); tv = tw["TVT"].to_numpy(float); tg = tw["GR"].to_numpy(float)
    o = np.argsort(tv); tv, tg = tv[o], tg[o]; _, ix = np.unique(tv, return_index=True)
    return tv[ix], tg[ix]
def misfit_curve(gr, true, calib_tvt, tv, tg):
    v = np.isfinite(gr)
    tgt = twgr(calib_tvt, tv, tg)
    coef, *_ = np.linalg.lstsq(np.vstack([gr[v], np.ones(v.sum())]).T, tgt[v], rcond=None)  # fit gain/offset
    cal = gr * coef[0] + coef[1]
    return np.array([np.mean((cal[v] - twgr(true + dz, tv, tg)[v]) ** 2) for dz in GRID])
def argmin_dz(gr, true, calib_tvt, tv, tg):
    return abs(GRID[int(np.argmin(misfit_curve(gr, true, calib_tvt, tv, tg)))])

orc, leg, shf, lat_span, tw_span, worst_idx = [], [], [], [], [], []
rs = np.random.RandomState(SEED)
for wi, w in enumerate(W):
    tv, tg = load_tw(w)
    if tv is None: continue
    gr = pd.Series(w["GR"][w["ei"]]).interpolate(limit_direction="both").to_numpy()
    true = w["true"]
    if np.sum(np.isfinite(gr)) < 20 or len(tv) < 10 or np.std(gr) < 1e-6: continue
    legal = w["last"] + (w["hwZ"][w["li"]] - w["Z"])          # legal flat-surface guess; Z is known on the tail (§1)
    orc.append(argmin_dz(gr, true, true, tv, tg))
    leg.append(argmin_dz(gr, true, legal, tv, tg))
    shf.append(argmin_dz(gr, true, true[rs.permutation(len(true))], tv, tg))
    lat_span.append(np.percentile(true,97)-np.percentile(true,3)); tw_span.append(np.percentile(tv,97)-np.percentile(tv,3))
    worst_idx.append(wi)
orc, leg, shf = map(np.array, (orc, leg, shf))
from IPython.display import display
_deg = pd.DataFrame({"median |dz| from truth (ft)": [np.median(orc), np.median(leg), np.median(shf)],
                     "localized within 2 ft": [np.mean(orc<=2), np.mean(leg<=2), np.mean(shf<=2)]},
                    index=["oracle (illegal)", "legal (public)", "shuffle (control)"])
_deg.index.name = "GR calibration"
display(_deg.style.format({"median |dz| from truth (ft)": "{:.2f}", "localized within 2 ft": "{:.0%}"})
        .apply(lambda r: ["background-color:#fde7e0"]*len(r) if r.name == "legal (public)" else [""]*len(r), axis=1))
print(f"context: the answer moves ~{np.median(lat_span):.0f} ft through a ~{np.median(tw_span):.0f} ft typewell column")

**Table 2 — the calibration degeneracy.** Median |dz| from the true surface and the fraction of wells localized within 2 ft, under three GR calibrations; the legal row (highlighted) sits at the random-shuffle level — the explicit gain/offset fit localizes only when calibrated at the truth.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.4))
labs = ["oracle\n(illegal)", "legal\n(public)", "shuffle\n(control)"]
frac = [np.mean(orc<=2), np.mean(leg<=2), np.mean(shf<=2)]
bars = ax[0].bar(labs, [f*100 for f in frac], color=[C["green"], C["red"], C["grey"]], width=.6)
for b, f in zip(bars, frac): ax[0].text(b.get_x()+b.get_width()/2, f*100+2, f"{f:.0%}", ha="center")
ax[0].set_ylabel("wells localized within 2 ft (%)"); ax[0].set_ylim(0, 100)
ax[0].set_title("The explicit fit pins the surface — only at the true calibration")
ax[1].barh(["typewell\ncolumn", "lateral\nanswer band"], [np.median(tw_span), np.median(lat_span)],
           color=[C["grey"], C["blue"]])
for i, v in enumerate([np.median(tw_span), np.median(lat_span)]): ax[1].text(v+10, i, f"{v:.0f} ft", va="center")
ax[1].set_xlabel("TVT span (ft)"); ax[1].set_title("…and the lateral barely moves through it")
plt.tight_layout(); plt.show()

Read the left panel. With the **oracle** calibration the GR-misfit minimum lands on the true surface in about 82% of wells (median offset 0 ft). The gamma-ray tracks lithology, as it should. But fit the same explicit gain/offset at any **legal** position and that skill collapses: only ~8% of wells localize, at or below the random-shuffle level, with the minimum wandering a dozen feet. The right panel says why this particular fit struggles. The answer moves through about 25&nbsp;ft while the typewell spans hundreds, so the lateral never travels far enough vertically to pin the gain and offset from the data alone.

To put a hard number on the absolute-positioning ceiling, I ran the most generous *illegal* version of GR matching there is — a separate train-side recompute, not a cell in this notebook: an oracle registration that sees the *true* eval gamma ray and grid-searches the best global offset and slope of the TVT line to match the typewell shape. Unlike the §2 line oracle, which fits the line to the true TVT, this one fits only the GR shape, and on a 200-well sample it scores about 31&nbsp;ft RMSE against ~17 for carrying the last value flat, recovering the eval slope at essentially zero correlation (−0.00). Letting GR alone choose the absolute position hurts, because GR(TVT) repeats: the best global match points confidently at a wrong depth. That is the same self-similarity §5 turns into the bimodal datum, and it is why the legal route in §5c pins the offset at the heel rather than fitting it globally. What GR carries is relative and coarse, best read sequentially against the typewell with the offset already pinned: that is the ~10 a particle filter reaches and the ~5 a better matcher reaches, not a global fit.

So the correction to my earlier claim. That confound is a property of *this explicit-calibration method*, not of the data. GR carries real coarse signal — a particle filter takes my selector from the ~16 floor down to ~10 with it (~10.9 under leave-whole-field-out), which is that tracker's level under a hard split, not a GR ceiling. And Tucker's ~5 per-well, as I read it, says a better matcher pulls more of the dip and wiggle out of the same data. The gap from my ~10 to ~5 is alignment headroom, not an information wall.

So the earlier framing — "GR cannot pin the slope, and that is a property of the data" — was too strong. Read this section as: the naive explicit gain/offset fit is confounded with position, and my tracker's ~10&nbsp;ft is *that tracker's* limit under one hard split. A sequential matcher that uses the typewell directly, the way the ~5&nbsp;ft solvers do, sidesteps the explicit fit and gets further.

Which sequential matchers actually get further? I ran three head-to-head on a fixed protocol — the same 150 train wells, the same grouped folds, each aligner's output fed to the same downstream model — and scored by the honest marginal delta it added once the confounds were stripped (separate train-side runs, not cells here). Only one survived. A windowed, shape-matching particle filter earned −0.18&nbsp;±&nbsp;0.04&nbsp;ft across five seeds, negative on all five. DTW with calibration was a wash overall and 1.2&nbsp;ft *worse* than flat on the wells with no real offset — it invents movement where there is none. Windowed cross-correlation made things slightly worse the moment its alignment features were stacked on two plain GR scalars. And the sharpest part is *where* the particle filter's value lives: not in the TVT it proposes (its median move is ~0.1&nbsp;ft, essentially staying home) but in its uncertainty — the posterior spread correlates with the actual error at +0.23. Stacked on a model as features, GR alignment earned its keep as a per-row trust gate, deciding when to believe the anchor, not as a predictor — the predicting in my own pipeline stayed with the bounded matcher (§8). FOYSAL's working note reaches the same boundary from the solver side: un-gated GR matching picks confident wrong depths, and the guard is where the value is.

One practical lever before the next section, from the Problem Breakdown thread: the GR sensor rotates as the tool turns in the hole (Shrey spotted it, Tabish hosted the breakdown), and an FFT of the GR shows the rotation frequency. MY0705's point is that you can use it to denoise the trace before matching, for cleaner input and better alignment.

<a id="5"></a>
## 5. The irreducible tail: a bimodal datum

If the recoverable part is offset plus dip read off the typewell, the irreducible part is where the typewell stops telling you the truth. This is the piece I think matters most.

The geology comes from souldrive, whose post laid it out (and who, I'll note, publicly endorsed the first version of this notebook — credit due both ways). The Eagle Ford is rhythmically bedded: Milankovitch orbital cyclicity drives limestone–marl couplets that repeat on a ~15–25&nbsp;ft scale. Because the bedding repeats, the horizontal GR can line up with the typewell at two stratigraphic positions about one bundle apart. The datum becomes bimodal, roughly ±15&nbsp;ft, and on a truly ambiguous well it is close to a coin-flip which mode is real. A minority of wells add their own version of this through sub-seismic faults, 15–30&nbsp;ft steps.

I came at the same object from the other side, on public train so a forker can reproduce it. Fit the smooth oracle per well and sort by squared error: the worst decile carries close to 40% of the total (the cell prints the exact figure), and my selector's own out-of-fold bias puts the same minority sharper still — the 63 of 355 wells with per-well bias over 8&nbsp;ft carry 73% of the error (an offline aggregate, not a cell here). On those worst wells, the §4 shift scan under the *oracle* calibration — the one that localizes 82% of normal wells — shows the GR misfit at the true TVT flat, or slightly worse than a match one bundle away. With the calibration that works elsewhere, that flat misfit is the data being ambiguous, not the fit failing: the typewell cannot tell the two positions apart.

There is a third sign the tail is where the leverage sits: it is the only place a different tree model buys anything. On an older 200-well cross-validation pass, swapping the XGB reference for an LGB+CatBoost blend bought about 0.8&nbsp;ft, winning all four folds, and the gain came off the two heavy-tail folds where XGB was the worst model. Those are directional deltas on a small, stale subset, not board numbers — but the direction is the point: the wells that carry the error are also the only wells where a different model earns its keep.

The decision theory is what makes this load-bearing, and pilkwang sharpened it in the comments. On a well with mass `p` at one position and `1−p` at the other a distance `d` apart, the squared-loss-optimal estimate is the posterior mean `p·a + (1−p)·b`, with irreducible risk `p(1−p)d²`; the midpoint I first reached for is just the `p = ½` case. Committing to one mode beats the midpoint only outside `¼ < p < ¾`, while the posterior mean does at least as well as either for every `p` — strictly better than the midpoint whenever `p ≠ ½`, and strictly better than a committed mode for every interior `p`. So the target isn't a binary bimodal flag feeding a fixed midpoint; it's a calibrated `p`, and the estimate leans toward whichever mode `p` favors.

And `p` falls out of machinery already on the page: the *relative depth* of the two minima in the §4 shift scan is a log-likelihood ratio between the modes, so a two-position softmax over those depths gives `p` directly. It needs a calibrated temperature, or `p` comes out overconfident — and there is a wrinkle that bites on exactly the hard wells: the GR likelihood there is flat or slightly inverted at the truth, the decoy one bundle away fitting a touch better, so a `p` read straight off the misfit leans toward the wrong mode. The irreducible loss is the variance of the datum; the practical question is how well you estimate `p` before that bias bites.

souldrive measured the same degeneracy from the cost-landscape side on all 773 wells: among the ~49% of wells with a clear second minimum in the bundle gap, the cost-favored one is the correct datum just 48.8% of the time, and the margin between the two carries essentially no information about which is right (r&nbsp;=&nbsp;+0.054, p&nbsp;=&nbsp;0.30). His NCC cost surface and the GR likelihood here agree: the tie is detectable, the tie-break absent. His [Decoding Eagle Ford](https://www.kaggle.com/code/souldrive/decoding-eagle-ford-why-some-wells-are-hard) notebook has the geology and the full decomposition.

Stack §3 through §5 together and the single "wall" dissolves into a distribution: a good chunk of wells near-solved from clean offset plus dip, a heavy tail that is geology — the bimodal datum and the faults — not a modeling failure. That recoverable-versus-irreducible split is the honest replacement for the ~10&nbsp;ft wall I claimed before.

A note on what is measured versus borrowed, so I don't over-correct one mistake into a new overclaim. The Milankovitch mechanism is souldrive's geology. The SSE concentration and the likelihood degeneracy are my train-side measurements, computed on the smooth-oracle residuals in the cell below; they need the truth, so they live on train, not on the hidden test. The midpoint-versus-mode arithmetic is decision theory on an idealized well. I am not claiming to have measured the bimodal *cause* on every tail well. The ±15&nbsp;ft picture beside the figure is a labeled schematic of the mechanism, not a measurement.

In [ ]:
# ---- §5, train-confirmed: where the error concentrates, and why those wells are ambiguous ----
order = np.argsort(sse_w)[::-1]                 # wells worst-first, by smooth-oracle SSE (from §2)
k10   = max(1, len(sse_w)//10)
share = sse_w[order[:k10]].sum() / sse_w.sum()
cum   = np.cumsum(sse_w[order]) / sse_w.sum()   # Lorenz: cumulative share of the error, worst-first
xf    = np.arange(1, len(cum)+1) / len(cum)

# the worst few wells' GR misfit vs a datum shift, under the ORACLE calibration (centred on truth):
worst_misfit = []                               # (well id, misfit / its own min) — the truth is at dz=0
for wi in order:
    if len(worst_misfit) >= 3: break
    w = W[wi]; tv, tg = load_tw(w)
    if tv is None: continue
    gr = pd.Series(w["GR"][w["ei"]]).interpolate(limit_direction="both").to_numpy()
    if np.sum(np.isfinite(gr)) < 20 or len(tv) < 10 or np.std(gr) < 1e-6: continue
    c = misfit_curve(gr, w["true"], w["true"], tv, tg)      # oracle calibration (at the truth)
    worst_misfit.append((w["wid"], c / c.min()))
print(f"worst 10% of wells carry {share:.0%} of the smooth-oracle squared error (train sample of {len(W)} wells)")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13.6, 3.7))

# (a) cumulative-SSE (Lorenz) curve — the concentration, measured on train
ax[0].plot(xf, cum, color=C["red"], lw=2)
ax[0].plot([0,1],[0,1], ls="--", lw=1, color=C["grey"], label="if error were uniform")
ax[0].scatter([0.10],[share], color=C["blue"], zorder=5)
ax[0].annotate(f"worst 10% of wells\ncarry {share:.0%} of SSE", (0.10, share),
               xytext=(0.34, max(share-0.22, 0.12)), fontsize=9,
               arrowprops=dict(arrowstyle="->", color=C["blue"]))
ax[0].set_xlabel("fraction of wells (worst first)"); ax[0].set_ylabel("cumulative share of SSE")
ax[0].set_title("Error is concentrated (train-confirmed)"); ax[0].legend(loc="lower right", fontsize=8)

# (b) GR misfit vs dz under the ORACLE calibration, for the worst few wells — truth not preferred
for wid, cn in worst_misfit:
    ax[1].plot(GRID, cn, lw=1.5, alpha=.85, label=wid)
ax[1].axvline(0, color="k", lw=.9, ls="--")
ax[1].set_xlabel("vertical shift dz from truth (ft)"); ax[1].set_ylabel("GR misfit (÷ its own min)")
ax[1].set_title("Worst wells: the truth isn't preferred"); ax[1].legend(fontsize=8, title="well")

# (c) schematic of the bimodal datum — an illustration of souldrive's mechanism, NOT a measurement
xx = np.linspace(-35, 35, 400); g = lambda m: np.exp(-0.5*((xx-m)/4.0)**2)
ax[2].fill_between(xx, g(-15), color=C["orange"], alpha=.55, label="mode A")
ax[2].fill_between(xx, g(15),  color=C["blue"],   alpha=.40, label="mode B (~1 bundle away)")
ax[2].axvline(0, color=C["green"], lw=2.2, label="posterior mean (midpoint at p=½)")
ax[2].set_yticks([]); ax[2].set_xlabel("TVT error (ft)")
ax[2].set_title("Schematic: bimodal datum (±15 ft)"); ax[2].legend(fontsize=7.5, loc="upper right")
plt.tight_layout(); plt.show()

**Does the hedge actually help? I tested pilkwang's recipe.** Centre the §4 shift scan on the truth and slide a constant datum offset — this centres on the truth, so it is a train-only diagnostic that measures whether the hedge *could* help, not a procedure you can run on the test set. The two deepest minima are the two candidate datums, and a two-position softmax over their depths gives `p`. The cell scores three datums on every well — committing to the global minimum, the posterior mean, and the fixed midpoint — under the *oracle* calibration, since §4b already showed the legal explicit fit barely localizes. The number is the constant datum error each leaves (0 = perfect; the bimodal gap is ~±15&nbsp;ft).

In [ ]:
def two_minima(cost, sep=6.0):
    i1 = int(np.argmin(cost)); dz1, m1 = GRID[i1], cost[i1]
    mask = np.abs(GRID - dz1) >= sep
    if not mask.any(): return dz1, m1, dz1, m1
    j = np.flatnonzero(mask)[int(np.argmin(cost[mask]))]
    return dz1, m1, GRID[j], cost[j]
def softmax_p(m1, m2, T):
    a = np.array([-m1 / T, -m2 / T]); a -= a.max(); e = np.exp(a); return e[0] / e.sum()

tail_set = set(order[:k10].tolist())                       # worst-decile wells, from the SSE sort above
err = {m: ([], []) for m in ["commit", "posterior", "midpoint"]}
loc_oracle = loc_legal = wrong = n = 0
for wi, w in enumerate(W):
    tv, tg = load_tw(w)
    if tv is None: continue
    gr = pd.Series(w["GR"][w["ei"]]).interpolate(limit_direction="both").to_numpy()
    if np.sum(np.isfinite(gr)) < 20 or len(tv) < 10 or np.std(gr) < 1e-6: continue
    legal = w["last"] + (w["hwZ"][w["li"]] - w["Z"])       # legal flat-surface guess (as in §4b)
    c_or = misfit_curve(gr, w["true"], w["true"], tv, tg)  # oracle calibration; dz=0 is the truth
    c_lg = misfit_curve(gr, w["true"], legal, tv, tg)      # legal calibration
    n += 1
    dz1, m1, dz2, m2 = two_minima(c_or)
    if abs(dz1) <= 2: loc_oracle += 1
    if abs(two_minima(c_lg)[0]) <= 2: loc_legal += 1
    if abs(dz1) > abs(dz2): wrong += 1
    p1 = softmax_p(m1, m2, 2.0 * m1 + 1e-9)            # T = 2*min_misfit ≈ 2σ², the Bayesian temperature
    pred = dict(commit=dz1, posterior=p1 * dz1 + (1 - p1) * dz2, midpoint=0.5 * (dz1 + dz2))
    nev = len(w["true"])
    for m in err:
        err[m][0].extend([pred[m]] * nev)
        if wi in tail_set: err[m][1].extend([pred[m]] * nev)
datum = {m: (rmse(err[m][0]), rmse(err[m][1])) for m in err}
print("datum error left (ft, oracle calibration):     all wells | worst-decile")
for m in ["commit", "posterior", "midpoint"]:
    print(f"  {m:<11} {datum[m][0]:>6.2f}     {datum[m][1]:>6.2f}")
print(f"\nglobal minimum localizes <=2 ft:  oracle {loc_oracle/n:.0%}   legal {loc_legal/n:.0%}"
      f"   (under oracle it still picks the wrong mode {wrong/n:.0%})")

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 3.3))
labs = ["commit", "posterior\n(pilkwang)", "midpoint"]
allv = [datum[m][0] for m in ["commit", "posterior", "midpoint"]]
tlv  = [datum[m][1] for m in ["commit", "posterior", "midpoint"]]
x = np.arange(3)
ax.bar(x-0.19, allv, 0.36, color=C["grey"], label="all wells")
ax.bar(x+0.19, tlv,  0.36, color=C["red"],  label="worst-decile tail")
for i, (a, t) in enumerate(zip(allv, tlv)):
    ax.text(i-0.19, a+0.08, f"{a:.1f}", ha="center", fontsize=8.5)
    ax.text(i+0.19, t+0.08, f"{t:.1f}", ha="center", fontsize=8.5)
ax.set_xticks(x); ax.set_xticklabels(labs); ax.set_ylabel("datum error left (ft)")
ax.set_title("The hedge vs committing (oracle calibration)"); ax.legend(fontsize=8.5)
plt.tight_layout(); plt.show()

The result is honest but modest. On the worst-decile tail the posterior mean leaves the least datum error of the three, beating both committing and the fixed midpoint — so the hedge helps where it should — but the tail margin is only ~0.7&nbsp;ft, so call it directional (over *all* wells committing edges it, 5.2 vs 5.7&nbsp;ft, because the temperature over-hedges the easy wells where the global min is already right). The caveat that matters most: the softmax `p` is a confidence weight, not a mode-fixer — it always backs the lower-misfit minimum, and when that is the decoy (12% of wells even under the oracle calibration) it backs the decoy too. The gain is from not fully committing, not from `p` picking the right mode. And all of this is the *oracle*-calibration ceiling; whether you can read `p` *legally* is the next question — §5c.

**Is the calibration the wall, or is there a legal route?** §4b's *legal* fit was a flat-surface guess, deliberately weak. The honest legal move is to fit the GR gain and offset on the **known heel**, where `TVT` is given, then carry that calibration to the tail; nothing there uses the answer. The cell localizes the datum under four calibrations: the two from §4b, the legal heel-fit, and the heel-fit after a GR-rotation denoise (a rolling median here, standing in for the FFT-rotation notch the Problem Breakdown thread proposed).

In [ ]:
def _calib(src, tgt):
    v = np.isfinite(src) & np.isfinite(tgt)
    if v.sum() < 20: return None
    c, *_ = np.linalg.lstsq(np.vstack([src[v], np.ones(v.sum())]).T, tgt[v], rcond=None); return c
def _denoise(g): return pd.Series(g).rolling(7, center=True, min_periods=1).median().to_numpy()
TW = {w["wid"]: load_tw(w) for w in W}            # cache typewells once

def datum_localize(mode, dn=False):
    # same GR prep + validity filter as §5b, so the oracle and flat rows reproduce its numbers exactly
    loc = n = 0
    for w in W:
        tv, tg = TW[w["wid"]]
        if tv is None: continue
        ge = pd.Series(w["GR"][w["ei"]]).interpolate(limit_direction="both").to_numpy()
        gk = pd.Series(w["GR"][w["kn"]]).interpolate(limit_direction="both").to_numpy()
        if dn: ge, gk = _denoise(ge), _denoise(gk)
        if np.sum(np.isfinite(ge)) < 20 or len(tv) < 10 or np.std(ge) < 1e-6: continue
        true = w["true"]
        if mode == "oracle":  coef = _calib(ge, twgr(true, tv, tg))
        elif mode == "flat":  coef = _calib(ge, twgr(w["last"] + (w["hwZ"][w["li"]] - w["Z"]), tv, tg))
        else:                 coef = _calib(gk, twgr(w["tvi"][w["kn"]], tv, tg))   # heel = LEGAL
        if coef is None: continue
        cal = ge * coef[0] + coef[1]; v = np.isfinite(cal)
        if v.sum() < 20: continue
        cost = np.array([np.mean((cal[v] - twgr(true + dz, tv, tg)[v]) ** 2) for dz in GRID])
        n += 1
        if abs(GRID[int(np.argmin(cost))]) <= 2: loc += 1     # truth is at dz=0
    return loc / max(n, 1)

print("GR-misfit datum localization (global min within 2 ft of the truth), by calibration:")
print(f"  oracle (illegal)          {datum_localize('oracle'):.0%}")
print(f"  flat-surface (§4b legal)  {datum_localize('flat'):.0%}")
print(f"  heel-fit (legal)          {datum_localize('heel'):.0%}")
print(f"  heel-fit + GR denoise     {datum_localize('heel', True):.0%}")

The flat fit localizes the datum on only ~8% of wells, but the legal heel-fit recovers ~80%, essentially the oracle's 82%, with the scan still centred on the true shape (a train-only diagnostic). The gap was the *calibration*, not a property of the data, and a GR-rotation denoise nudges it to 84% (a 7-point rolling median, a crude proxy for the proper FFT-rotation notch). So pilkwang's recipe has a legal route after all: heel-calibrate (the piece I add), then read the two minima and hedge (pilkwang's). I owe a correction to my §5 caveat, which leaned on the weak flat fit. The datum is legally pinnable to nearly the oracle's rate with an explicit fit; the calibration is no longer the bottleneck, and you do not need a separate matcher for it. The residual wrong-mode rate, about one well in eight, is unchanged by calibration, because that part is the geology, not the fit. What the legal route still cannot pin is the *shape*, the per-well dip the heel line cannot extrapolate to the tail (§6). The open legal problem moved from the datum to the slope.

<a id="6"></a>
## 6. The leave-field-out result, in proportion

This is the experiment that led me to the wrong "wall" claim, so let me run it honestly and then scope it correctly.

The setup is conservative by design — it is the **wall test** from §9, pointed at my own features. Cluster the wells by location (KMeans on heel X/Y), then GroupKFold so each fold leaves whole fields out, and try to predict the per-well drift slope from cheap legal features. The result: field-grouped OOF R² below zero — worse than guessing the global mean — and the shuffled-label control sits in the same place, so the harness itself is not leaking.

CNN, GRU and GBM all failed to beat the selector, and I mean I built them, not hand-waved: a seq2seq with multi-head cross-attention over the well's 401-token typewell memory bank, a dilated 1-D CNN with a ~1000-sample receptive field, a bidirectional GRU, and a LightGBM on band-limited typewell-inversion features. None cleared the per-well selector — though the seq2seq never fully converged, so I won't claim that door is 100% shut. What made me stop was an external, checkable benchmark: hengck23, a Grandmaster, was at rank&nbsp;163 with 7.583 after 26 submissions on a self-trained CNN/MTP/SDF stack — below the then-best free public fork at 7.519. If a Grandmaster's heavy-ML lands under the fork wave, my self-trained heavy-ML had negative expected edge for a bronze target, so I put the compute back into alignment quality. The board's own discussion has since converged on the same split: Tucker says the sub-6 CVs come from non-tabular models, and k256 reports his best non-tabular single method at 7.098 against a tabular best of 6.798 — the frontier is the matching, not the model class.

There is a clean way to see why the slope will not transfer. The TVT *level* is strongly coherent in space — a well's last-known TVT predicts its nearest neighbour's at correlation 0.976 — but that coherence is already yours for free, pinned by the heel. The quantity you actually have to predict, the in-zone *drift*, has nearest-neighbour correlation −0.08, essentially zero (both 265-well recomputes, not cells here). The spatial structure that exists is the part the anchor already hands you; the part you need is structureless from one well to the next.

wharekawa, who reproduced the §2 ladder, put a mechanism on this failure in the comments: reconstruction error scales like the surface gradient times the distance to the wells you borrow from. My recompute agrees (separate, not a cell here). Between same-field nearest neighbours — median spacing ~490&nbsp;ft, p90 ~1.8&nbsp;kft — the surface level already differs by ~13&nbsp;ft at the median and ~79&nbsp;ft at the p90, an implied gradient of ~0.023&nbsp;ft/ft (p90 ~0.075). That is a survey-density floor, not a modeling failure: the bounded IDW below (~23&nbsp;ft) sits right on it, and the 13–129&nbsp;ft band my leaky v1 produced is the same law read at wider effective spacing. The plane fit's blow-ups are the one thing the law does not cover — that part is ill-conditioning, not geology.

Cross-well surface reconstruction fails the same way. A bounded inverse-distance average of the neighbours' surface reconstructs to about 23&nbsp;ft RMSE — still worse than the ~14.5 of carrying the last value flat on those wells. Fitting a local dipping *plane* instead does not merely fail, it diverges numerically: the trajectory is nearly one-dimensional, so the neighbours are almost collinear, the dip terms are ill-conditioned, and the extrapolation flies off (a leak-fixed re-run lands at 477&nbsp;ft within a field and 3254&nbsp;ft across fields — a blow-up of the ill-conditioned plane fit, not a geological number). An earlier version reported a gentler 13–129&nbsp;ft; that was a same-field-leak artifact, its "leave-field-out" neighbours nearly all from the same field. Either way the cross-field reconstruction does not beat flat. (These are separate train-side re-runs, not cells in this notebook.)

The finding is real: the per-well drift slope is not learnable from the cheap legal features I tried when you generalize to unseen fields. I stand by that.

What I got wrong was the scope. This is leave-*whole-field*-out, deliberately the hardest split I could pose. The competition does not ask you to extrapolate to brand-new fields, and I should not assume more about the test geometry than the host has confirmed. My selector's honest level was ~10 pooled (~10.9 under this block-CV) — its ceiling under one hard split, not the task's floor. Tucker's ~5 per-well, as I read it, goes well below that without ever touching the cross-field slope transfer I proved hard.

So state the boundary cleanly, because both facts are true and answer different questions. The negative result says cross-field slope transfer is hard from cheap features; that is real. It does not say the per-well dip is unreadable — the board heads and my own smooth oracle at ~3.0 say the opposite — and it is not a within-field learnability claim either way.

The useful caution survives intact: a within-sample slope fit looks predictive and evaporates on hold-out, so "I found a feature that predicts the drift" usually means "I leaked." Still true. It just was not the floor of the task.

In [ ]:
# per-well legal features (known at prediction time) + the target slope
rows = []
for w in W:
    li, kn = w["li"], w["kn"]; Zf, MDf, X, Y = w["hwZ"], w["hwMD"], w["X"], w["Y"]
    ancc_kn = w["tvi"][kn] + Zf[kn]; latd_kn = np.hypot(X[kn]-X[li], Y[kn]-Y[li])
    pre_dip = np.polyfit(latd_kn, ancc_kn, 1)[0] if latd_kn.max() > 50 else 0.0
    g = lambda a: np.gradient(a, MDf)[li]
    rows.append(dict(wid=w["wid"], x=X[li], y=Y[li],
                     slope=np.polyfit(w["s"], w["true"]-w["last"], 1)[0],
                     pre_dip=pre_dip, dz_heel=g(Zf),
                     incl=np.degrees(np.arctan2(np.hypot(g(X), g(Y)), abs(g(Zf))+1e-9)),
                     gr_pre=np.nanmean(w["GR"][kn][-50:]),
                     latspan=np.hypot(X[w["ei"][-1]]-X[li], Y[w["ei"][-1]]-Y[li])))
D = pd.DataFrame(rows)
N_FIELDS = 17                                          # match the cited block-CV (KMeans-17 on heel X/Y)
XY = D[["x","y"]].values; XYs = (XY-XY.mean(0))/(XY.std(0)+1e-9)
field = KMeans(N_FIELDS, n_init=10, random_state=SEED).fit(XYs).labels_   # spatial field clusters
feats = ["pre_dip","dz_heel","incl","gr_pre","latspan","x","y"]
y = D["slope"].values
r2 = lambda yt,yp: 1 - np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2)
oof, oof_shuf = np.zeros(len(D)), np.zeros(len(D)); yshuf = rng.permutation(y)
for k in range(N_FIELDS):
    tr, te = field!=k, field==k
    if te.sum()==0 or tr.sum()<15: oof[te]=oof_shuf[te]=y[tr].mean() if tr.sum() else y.mean(); continue
    mk = lambda yy: HGB(max_iter=250,max_depth=3,learning_rate=.05,l2_regularization=2.,random_state=SEED).fit(D.loc[tr,feats],yy[tr]).predict(D.loc[te,feats])
    oof[te], oof_shuf[te] = mk(y), mk(yshuf)
print(f"drift-slope field-OOF R² = {r2(y,oof):+.3f}")
print(f"shuffle-control     R² = {r2(y,oof_shuf):+.3f}   (≤ 0, same as the real fit ⇒ no spurious skill in the harness)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.5))
ax[0].scatter(y, oof, s=14, alpha=.6, color=C["blue"])
lim = np.percentile(np.abs(y), 98)
ax[0].plot([-lim,lim],[-lim,lim], color=C["grey"], ls="--", lw=1, label="ideal (y = x)")
ax[0].set_xlim(-lim,lim); ax[0].set_ylim(-lim,lim)
ax[0].set_xlabel("true drift slope"); ax[0].set_ylabel("predicted (field hold-out)")
ax[0].legend(loc="upper left", fontsize=8, framealpha=.9)
ax[0].set_title(f"No skill on unseen fields  (R²={r2(y,oof):+.2f})")
labels = ["drift slope\n(field-OOF)", "shuffle\ncontrol"]
ax[1].bar(labels, [r2(y,oof), r2(y,oof_shuf)], color=[C["red"], C["grey"]], width=.5)
ax[1].axhline(0, color="k", lw=.8); ax[1].set_ylabel("R²  (>0 ⇒ learnable)")
ax[1].set_title("Both sit at/below zero")
plt.tight_layout(); plt.show()

<a id="7"></a>
## 7. Traps that look like progress

A few things on this task feel like progress without being it. Most are about how results are read; the last one I walked into myself.

**The CV→LB mirage.** A deep model can post a low validation RMSE on train folds and land far worse when it runs for real. At least one public notebook here advertises a single-digit fold RMSE yet scores around 14 when actually run. A low cross-validation number is a claim about *seen* wells; the leaderboard is a claim about *unseen* ones, and on this task they come apart hard. Validate on a hold-out shaped like the real test, not the friendliest fold.

**Seed and refork variance.** The strong public pipeline contains a stochastic particle filter. Re-running the same code, or forking a notebook whose public score is a few thousandths better, draws a fresh sample, and the run-to-run spread is comparable to the fork-to-fork gaps people chase (the real cluster lives around 7.15–7.5). pilkwang put a clean number on this in the comments: his byte-identical notebooks scored 7.201–7.286 on the public board, and a wider set identical up to comment encoding reaches down to 7.168 — the whole spread is the particle filter reseeding, with nothing changed in the code. So "a fork that reads 0.03 better than its neighbor" is just the lucky tail of that spread, and picking your final submission by best public score is selecting on it. Before celebrating a small delta, check whether it clears the seed band.

Stacking the public forks has its own ceiling. fle3n's public blend diagnoses it: the two strongest public engines carry error-correlation ρ ≈ 0.89, so the optimal blend weight sits near ½ and the blended RMSE is flat to under 0.001&nbsp;ft around it (it lands near 7.57). That is his estimate from three leaderboard anchors rather than a board fact, but the point holds — two engines whose errors correlate about 0.89 leave almost nothing for a re-weighting to recover. More signal needs a *decorrelated* source, not another re-tune of the same two.

**Chasing the ambiguous tail.** This is the counterintuitive one. On a bimodal well (§5), committing to a mode — the move that feels like finally *solving* that well — raises your expected error versus the midpoint hedge (about 21 vs 15 on the idealized 50/50 well). So apparent progress on the hardest wells can run in the wrong direction. Spend the effort where the error is recoverable, and hedge the tail.

And a meta-trap, since I fell into it: a deliberately conservative validation split can over-reject. My leave-field-out told me "unlearnable," and I generalized that to "the task's floor." Match the split to the test the competition actually scores, or a pessimistic guardrail turns into a false ceiling.

<a id="8"></a>
## 8. What to do if you are stuck near the cluster

- **The leverage is the median wells, not the tail.** Clean structural smoothing of the offset plus piecewise dip, and better per-well GR alignment to the typewell — that is where the gap from the fork cluster down to the heads lives, and it is per-well, not cross-field. A sizable share of wells can reach 3–5&nbsp;ft (the §2 cell prints how many).
- **Estimate the mode weight, predict the posterior mean.** On a bimodal well the variance is irreducible, so the right estimate is `p·a + (1−p)·b`, not a commitment to one mode (§5). The detector and the hedge collapse into one object: read `p` via the two-position softmax over the relative depths of the two minima in the §4 shift scan, as in §5. §5c shows that a legal heel-calibration makes those depths trustworthy: it recovers the oracle localization, so the datum needs no separate matcher; what the legal route still cannot pin is the per-well shape (§6). The estimate then moves off the midpoint toward whichever mode `p` favors as `p` leaves ½. A particle filter kept from collapsing to one mode carries that `p` and emits the posterior mean for free — which is also why a blanket jump-guard backfires here, since suppressing the big jump deletes the second mode.
- **Spend a hold-out budget before a tuning budget.** Validate on a split shaped like the real test, and keep the harsh leave-whole-field-out as a pessimistic guardrail rather than the target. If you can construct a within-field hold-out, run it too as a second, less-pessimistic reading — but neither split is confirmed to match the hidden test geometry, so treat them as bracketing it. Run a seed sweep before trusting a small delta; an afternoon of hold-out budget beats a week of tuning.
- **Drop the confirmed dead end.** Cross-well surface reconstruction across fields fails badly — the dip-plane fit blows up on the near-collinear neighbours, and even a bounded IDW average cannot beat flat (§6). Within-well typewell matching transfers; the cross-field offset does not.
- **Bound any GR-to-typewell search, or it will lie to you confidently.** Real per-well excursions are ~15–30&nbsp;ft, so give a matcher a window of about ±60&nbsp;ft around the last known TVT and no more. Let it roam much further and the §4 self-similarity takes over: a naive anchored Viterbi given a ±140&nbsp;ft band scored 29 RMSE against 15 for carrying flat, and a slope-following particle filter blew up to 34–58 across its whole tuning sweep — every variant allowed to wander lost to doing nothing. The bounded per-well path was also the field-robust one: it beat flat on all eight held-out fields (+2.75 to +8.34&nbsp;ft), while the three obvious far-zone patches (shrinking the far zone, a distance-ramped hold, cross-scale averaging) all died on held-out fields. (Separate train-side runs, not cells here; my own public-LB optimum was an interior blend, so read the all-fields win as robustness, not a board claim.) And the look-ahead is legal (Tiago thread), so whole-trace matching is allowed — use it.

<p style="background-color:#eef6ff;padding:16px;color:#111;font-size:15px;border:2px solid #9ec5ff;border-radius:6px">
<strong>If you keep one thing:</strong> the floor is not ~10. The board heads and my own smooth oracle (~3.0) both sit well below it, so the recoverable part runs much deeper than my selector reached. Spend your effort on median-well GR alignment and dip smoothing, and on the bimodal tail estimate the mode weight and predict the posterior mean rather than committing to one mode.
</p>

<a id="9"></a>
## 9. Fork the ruler, not the model

Fork the top pipeline for a score 0.03 better than its neighbour and you have forked the particle filter's lucky seed, not an edge ([§7](#7)) — that is what forking the *model* gets you. The **ruler** is the other thing here — and it has nothing to do with my predictions. These three helpers read *your* regression, not mine.

`oracle_ceiling(wells)` takes your per-group `(true, s)` and returns the constant / line / smooth RMSE floor for *your* data — your version of the §2 ladder. It answers whether your latest gain is real structure or noise below the floor.

`tail_concentration(wells)` fits the smooth oracle per group and returns the worst-10% share of the total squared error — the number that decides whether to chase the median or the tail. Your figure will differ from §5; that one is my train sample, this reads your tail.

The **wall test** (`wall_test(D, feats)`) leaves a whole group out and checks whether your feature survives, with a shuffled-label control so you know the test itself isn't the leak. Read it two ways. A feature that looks predictive in-sample and dies under the wall test was leaking — that is the check working. A feature that survives a within-group split but dies leave-group-out is fighting extrapolation, harder than what the competition scores, so don't read it as a verdict on the floor. Pair it with a within-group or random-row split to see both the harsh and the realistic number.

None of it is ROGII. A "well" is just a group; "measured depth" is just a within-group position; and the wall test takes **any group column** — spatial fields are one case, cross-validation folds and entity ids are others (pass `group_col=`; it clusters heel coordinates only as a fallback). If a feature you love dies under the wall test, that is the tool doing its job, and better to find out now than on your private split.

I am not asking you to fork this. I am telling you what forking is for.

In [ ]:
# ---- reusable harness: drop your own per-well (true, pred) in and read the ceiling / tail / wall ----
def oracle_ceiling(wells):
    # wells: list of dicts with 'true' (eval truth) and 's' (normalized MD 0..1). Returns the ceiling ladder.
    E = {k: [] for k in ["constant", "line", "smooth"]}
    for w in wells:
        t, s = np.asarray(w["true"], float), np.asarray(w["s"], float)
        E["constant"].append(t - t.mean())
        E["line"].append(t - np.polyval(np.polyfit(s, t, 1), s))
        E["smooth"].append(t - robfit(s, t, 6))
    return {k: rmse(np.concatenate(v)) for k, v in E.items()}

def tail_concentration(wells, frac=0.10):
    # worst-frac share of total smooth-oracle SSE — your version of the §5 tail.
    # your share will differ from the §5 figure (that's my train sample); this reads YOUR tail.
    sse = np.array([float(np.sum((np.asarray(w["true"], float) - robfit(np.asarray(w["s"], float), np.asarray(w["true"], float), 6))**2)) for w in wells])
    o = np.argsort(sse)[::-1]; k = max(1, int(len(sse)*frac))
    return round(float(sse[o[:k]].sum()/sse.sum()), 3)

def wall_test(D, feats, target="slope", group_col=None, n_fields=17, seed=42):
    # The HARD leave-one-GROUP-out test + a leak check, NOT a verdict on the task floor.
    # D: one row per group, with feature columns + a target column, and EITHER a group_col
    #    (fold id, entity id, time block, field...) OR x,y coordinates that get clustered into
    #    n_fields pseudo-fields when group_col is None. Group-out OOF R^2 (+ shuffle control).
    # Pair with a within-group or random-row split to get the realistic (less pessimistic) number.
    if group_col is not None:
        fld = pd.factorize(D[group_col].values)[0]; n_fields = int(fld.max()) + 1
    else:
        XY = D[["x","y"]].values; XYs = (XY-XY.mean(0))/(XY.std(0)+1e-9)
        fld = KMeans(n_fields, n_init=10, random_state=seed).fit(XYs).labels_
    y = D[target].values; rg = np.random.RandomState(seed); ys = rg.permutation(y)
    oof = np.zeros(len(D)); oos = np.zeros(len(D))
    for k in range(n_fields):
        tr, te = fld!=k, fld==k
        if te.sum()==0 or tr.sum()<15: oof[te]=oos[te]=y[tr].mean() if tr.sum() else y.mean(); continue
        f = lambda yy: HGB(max_iter=250,max_depth=3,learning_rate=.05,l2_regularization=2.,random_state=seed).fit(D.loc[tr,feats],yy[tr]).predict(D.loc[te,feats])
        oof[te], oos[te] = f(y), f(ys)
    r2 = lambda a,b: 1-np.sum((a-b)**2)/np.sum((a-a.mean())**2)
    return dict(field_oof_r2=round(r2(y,oof),3), shuffle_r2=round(r2(y,oos),3))

def posterior_mean(p, pred_lo, pred_hi):
    # squared-loss-optimal estimate on a two-mode well (pilkwang): p*lo + (1-p)*hi.
    # p = P(low mode) read from the two §4 shift-scan minima via a two-position softmax;
    # midpoint is the p=0.5 special case. apply only where a second mode is actually present.
    p = np.asarray(p, float)
    return p*np.asarray(pred_lo, float) + (1.0-p)*np.asarray(pred_hi, float)

# design notes: the oracle ladder is train-only / pooled row-RMSE; the reference lines are the REAL
# public LB (15.883 / ~7.2 / heads ~5.3-5.5 as of Jul 3); the field-OOF is a negative result on cheap features under a hard
# leave-field-out split; numbers shift a few hundredths across seeds/subsamples; no submission here.
print("ceiling:", {k: round(v,2) for k,v in oracle_ceiling(W).items()})
print("tail   :", tail_concentration(W), "(worst-10% share of SSE)")
print("wall   :", wall_test(D, feats))

<a id="10"></a>
## 10. Design choices and limitations

- **The oracle ladder is on train wells** (it needs the truth) and is *pooled* row-RMSE on a 250-well sample. The leaderboard reference lines are the real, honestly-scored public LB (carry-last 15.883, forks ~7.2, heads ~5.3–5.5 — top three 5.26/5.44/5.48, checked 2026-07-03). Bars and dashed lines are separate categories on a shared scale; the qualitative ordering carries the argument. The agreement between my ~16 flat oracle and the 15.883 baseline is a scale sanity check, not a validation — different well sets, same procedure.
- **The leave-field-out result is a negative result for the cheap features under a field-grouped split.** A fundamentally different signal could exist; the evidence says the cheap ones do not transfer across fields. It answers the harder cross-field question, not within-field learnability.
- **The bimodal-tail numbers are part measured, part borrowed.** The Milankovitch mechanism is souldrive's. The SSE concentration (worst-10% share) and the likelihood degeneracy are my own train-side measurements on the smooth-oracle residuals; the printed numbers are sample-specific and will shift on other data. The companion split — 63 of 355 wells with per-well bias over 8&nbsp;ft carrying 73% of the error — is read from my selector's own out-of-fold predictions, an aggregate computed offline (the per-well CSV is not shipped); it is a train-side measurement, not the hidden test. The 0.976 / −0.08 neighbour correlations, the dip-plane re-run and the neighbour-spacing/gradient estimator (§6), the oracle GR-registration and the aligner tournament (§4), and the bounded-window field-CV runs (§8) are likewise separate train-side recomputes, not cells here. fle3n's engine correlation (ρ≈0.89, §7) is his own published estimate from three leaderboard anchors, not mine. The midpoint-versus-mode result is decision theory on an idealized 50/50 well. None of it is measured on the hidden test.
- **The public-LB fraction is approximate.** Roughly 26% of the ~200 hidden wells, on the order of 50; the private set decides medals; the host excluded one outlier private well.
- **On the retracted leak.** The visible `data/test/` example wells are byte-identical to the first three train wells, and an ANCC-cheat scored ~0.007 against that example set. That is what copies would do, and it is what I misread as the public LB being scored on copies. The real public LB is the hidden wells.
- **Subsample and seed.** Numbers shift a few hundredths across seeds and sample sizes; the ordering and signs are stable. Re-run with `N_WELLS = 773` to confirm.
- **No submission here.** This notebook scores nothing and is not meant to.

<a id="11"></a>
## 11. References and a question for you

The corrected thesis leans on more of the community than the first version did. Each of these is worth an upvote.

**The structure of the task**
- [franticXu — formation columns are derived from the typewell, not independent 3D surfaces](https://www.kaggle.com/competitions/rogii-wellbore-geology-prediction/discussion/708167): the one-degree-of-freedom result with the thickness evidence, and the point that `TVT` is cumulative vertical distance, not a single-layer thickness.
- [Tabish — Problem Breakdown](https://www.kaggle.com/competitions/rogii-wellbore-geology-prediction/discussion/708367): the typewell as a GR-vs-TVT lookup, shared subsequences of a master sequence, the piecewise-linear dip ("parallel jagged lines"), and the GR-rotation denoising lever (Shrey first spotted the rotation, MY0705 proposed the FFT denoise).
- [pilkwang — EDA: target-free alignment for TVT](https://www.kaggle.com/code/pilkwang/rogii-eda-target-free-alignment-for-tvt): the `TVT = −Z + S(X,Y) + b` framing and the GR-residual scale, with a careful look at leakage and overlap. In the comments here he also turned the §5 hedge into a calibrated posterior mean and supplied the seed-variance numbers cited in §7.

**The irreducible tail**
- [souldrive — the ±15 ft datum](https://www.kaggle.com/competitions/rogii-wellbore-geology-prediction/discussion/711878): the Milankovitch / bimodal-datum geology that §5 is built on. souldrive also publicly endorsed the first version of this notebook; the §5 synthesis is partly his.

**Starters and where a real pipeline's error lives**
- [Chris Deotte — EDA Starter](https://www.kaggle.com/code/cdeotte/eda-starter) and [XGB Starter](https://www.kaggle.com/code/cdeotte/xgb-starter-cv-15): the canonical orientation and a clean grouped-CV baseline.
- [mitchgansemer — GR features & outlier detection](https://www.kaggle.com/code/mitchgansemer/gr-features-outlier-detection-rogii-wellbore) and [drift-targeting](https://www.kaggle.com/code/mitchgansemer/drift-targeting-ncc-tree-based-rogii-wellbore): where the error concentrates on the long-tail wells, a useful counterpoint to §5 and §7.

**Rules and facts that settled things**
- [FOYSAL — working note: anchors, GR alignment, and guarded geosteering](https://www.kaggle.com/writeups/foysalemonshanto/rogii-wellbore-prediction-anchors-gr-alignment): a solver-side account that converges on the same guardrails — un-gated GR matching fails, guarded search is the fix (§4, §8).
- [k256 — Score Without Tabular Models](https://www.kaggle.com/competitions/rogii-wellbore-geology-prediction/discussion/717573): where the non-tabular frontier became explicit — his 7.098 non-tabular single method vs 6.798 tabular best, and Tucker's note that the sub-6 CVs are non-tabular (§6).
- wharekawa, who reproduced the §2 oracle ladder to the last digit and proposed the §6 survey-density mechanism in the comments of the discussion topic.
- The Tiago thread, where within-well look-ahead (using the full per-well GR trace) was confirmed legal, so whole-trace typewell matching is fair.
- Tucker (rank 2) described reaching about 5 pooled per-row RMSE using per-well data only — a train-CV figure, not a leaderboard score. His ~5 is his own stated self-report, not a shared notebook, and it is the corroboration that the recoverable part runs below ~10.
- Ioannis M (rank 28) and the host posts that settled the test-set question: the visible `data/test/` is example data, replaced by the real hidden test.

If you want the floor and the field-extrapolation numbers on your own predictions, the helpers in §9 are self-contained — fork and point them at your OOF.

---
### Credits and a question for you

Thanks to the organizers and to franticXu, Tabish, pilkwang, souldrive, wharekawa, Chris Deotte and mitchgansemer, whose public work this builds on. License: Apache 2.0.

I got two things wrong in the first version, the leak and the wall, and I have tried to map the task more honestly here: a recoverable part (per-well offset plus piecewise dip, read off the typewell, where the median wells live) and an irreducible part (the bimodal datum on the tail, where the posterior mean beats any single mode). The floor is not ~10; the board heads and the smooth oracle at ~3.0 both sit well below it. What I would like to see, and will run the harness on:

- a leak-free hold-out shaped like the real test, that I can validate against;
- a better GR alignment that pushes the median wells toward ~5;
- a reliable way to read the mode weight `p`, so the posterior mean lands only where a second mode is real.

Drop it in the comments. If you disagree, bring a hold-out and I'll run it.